# 2da Practica Calificada CC521 - Computacion Evolutiva

**Alumno:** Luis Antonio Rojas Huaroc  
**Codigo UNI:** 20171581B  
**Profesor:** LANH

Resuelvo con lo del lab dirigido, el script GA de clase (Semana 5) y lo visto de esquemas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(20171581)

## Pregunta 1

El espacio tiene 2097152 puntos. Como es AG binario:

$$2^l = 2097152 \Rightarrow l = 21$$

Cada individuo es una cadena de 21 bits.

In [ ]:
N_puntos = 2097152
l = int(np.log2(N_puntos))
print("l =", l)
print("chequeo 2^l =", 2**l)

### a) Numero total de esquemas

Cada punto del espacio contiene $2^l$ esquemas (ver parte c).
Como hay $2^l$ puntos, si cuento el total sumando lo de cada punto:

$$\text{total} = 2^l \cdot 2^l = 2^{2l} = 2^{42}$$

(Si se hablara solo de esquemas distintos seria $3^l$, pero aqui piden el total sobre el espacio.)

In [ ]:
total_esquemas = (2**l) * (2**l)
print("a) Numero total de esquemas =", total_esquemas)
print("   (2^(2l) = 2^42)")

### b) Numero total de puntos de busqueda

Es el dato del enunciado:

$$2^l = 2^{21} = 2097152$$

In [ ]:
total_puntos = 2**l
print("b) Numero total de puntos de busqueda =", total_puntos)

### c) Numero de esquemas contenidos en cada individuo

En un cromosoma fijo, en cada posicion puedo dejar el bit o poner `*`.
Son 2 opciones por posicion:

$$2^l = 2^{21}$$

In [ ]:
esquemas_por_individuo = 2**l
print("c) Esquemas por individuo =", esquemas_por_individuo)

### d) Cotas inferior y superior (poblacion n = 50)

- Cota inferior: si los 50 son iguales, solo hay los esquemas de uno: $2^l$
- Cota superior: si aportan lo maximo, $n \cdot 2^l$

$$2^l \leq \#\text{esquemas} \leq n \cdot 2^l$$

In [ ]:
n_pob = 50
cota_inf = 2**l
cota_sup = n_pob * (2**l)
print("d) Cota inferior =", cota_inf)
print("d) Cota superior =", cota_sup)

### e) Cota inferior del numero esperado de individuos (de t a t+1)

Aqui piden **individuos**, no esquemas.
En el AG simple la poblacion se mantiene en n: de la generacion t a la t+1 se obtienen n individuos.

Con $n = 50$:

$$\text{cota inferior} = 50$$

In [ ]:
cota_e = n_pob
print("e) Cota inferior individuos (t -> t+1) =", cota_e)

### Resumen pregunta 1

In [ ]:
print("===== RESUMEN P1 =====")
print("l =", l)
print("a) total esquemas        =", total_esquemas)
print("b) total puntos          =", total_puntos)
print("c) esquemas / individuo  =", esquemas_por_individuo)
print("d) cota inferior         =", cota_inf)
print("d) cota superior         =", cota_sup)
print("e) individuos t->t+1     =", cota_e)

## Pregunta 2

Solo piden diseno de codificacion (no dan la f para optimizar).

| Variable | Rango | Precision |
|---|---|---|
| x | [-20.0, 125.0] | 0.5 |
| y | [0, 1.2e6] | 10000 |
| z | [-0.1, 1.0] | 0.001 |

Codificacion de punto fijo, mapeada y concatenada: `[bits_x | bits_y | bits_z]`.

Bits por variable:

$$\frac{b_i - a_i}{2^{l_i}-1} \leq \text{precision}_i \quad\Rightarrow\quad 2^{l_i}-1 \geq \frac{b_i-a_i}{\text{precision}_i}$$

In [ ]:
def bits_necesarios(a, b, precision):
    necesito = (b - a) / precision
    li = 0
    while (2**li - 1) < necesito:
        li = li + 1
    paso = (b - a) / (2**li - 1)
    return li, necesito, paso

lx, nx, pasox = bits_necesarios(-20.0, 125.0, 0.5)
ly, ny, pasoy = bits_necesarios(0.0, 1.2e6, 10000.0)
lz, nz, pasoz = bits_necesarios(-0.1, 1.0, 0.001)

print("x: rango/prec =", nx, "; lx =", lx, "; paso =", pasox)
print("y: rango/prec =", ny, "; ly =", ly, "; paso =", pasoy)
print("z: rango/prec =", nz, "; lz =", lz, "; paso =", pasoz)
print("Total bits =", lx + ly + lz)

Me salio:

- $l_x = 9$ ($2^8-1=255 < 290$, $2^9-1=511 \geq 290$)
- $l_y = 7$ ($2^6-1=63 < 120$, $2^7-1=127 \geq 120$)
- $l_z = 11$ ($2^{10}-1=1023 < 1100$, $2^{11}-1=2047 \geq 1100$)

**Minimo de bits = 9 + 7 + 11 = 27**

Mapeo:

$$x = a_x + \frac{\mathrm{int}(bits_x)}{2^{l_x}-1}(b_x-a_x)$$

(igual para y, z). Para codificar redondeo al entero mas cercano.

In [ ]:
def codificar(valor, a, b, li):
    entero = (valor - a) / (b - a) * (2**li - 1)
    entero = int(round(entero))
    if entero < 0:
        entero = 0
    if entero > (2**li - 1):
        entero = 2**li - 1
    cadena = format(entero, '0{}b'.format(li))
    return entero, cadena

def decodificar(entero, a, b, li):
    return a + entero / (2**li - 1) * (b - a)

def codificar_punto(x, y, z):
    ix, sx = codificar(x, -20.0, 125.0, lx)
    iy, sy = codificar(y, 0.0, 1.2e6, ly)
    iz, sz = codificar(z, -0.1, 1.0, lz)
    return sx + sy + sz, (ix, iy, iz), (sx, sy, sz)

# el segundo punto lo tomo como (125.0, 1.2e6, 1.0)
puntos = [
    (-20.0, 0.0, -0.1),
    (125.0, 1.2e6, 1.0),
    (50.0, 100000.0, 0.597),
]

for p in puntos:
    crom, enteros, bloques = codificar_punto(*p)
    print("Punto", p)
    print("  enteros =", enteros)
    print("  x,y,z   =", bloques[0], bloques[1], bloques[2])
    print("  cromosoma 27 bits =", crom)
    xd = decodificar(enteros[0], -20.0, 125.0, lx)
    yd = decodificar(enteros[1], 0.0, 1.2e6, ly)
    zd = decodificar(enteros[2], -0.1, 1.0, lz)
    print("  decodificado ~", (xd, yd, zd))
    print()

### Cadenas (pregunta 2)

1. $(-20,0,-0.1)$ -> `000000000000000000000000000`
2. $(125, 1.2\times10^6, 1.0)$ -> `111111111111111111111111111`
3. $(50, 100000, 0.597)$ -> `011110111000101110100010001`
   - x: 247 -> `011110111`
   - y: 11 -> `0001011`
   - z: 1297 -> `10100010001`

## Pregunta 3

Maximiziar $f(x) = 3x^{10}$ con el **algoritmo genetico simple usado en clase** (script GA basico).

### Justificacion de parametros

- **Dominio:** el enunciado no lo da. Tomo $x \in [0,1]$ porque si no el maximo se va a infinito; ahi el optimo es $x=1$, $f=3$.
- **precision:** $0.001$. Entonces $\frac{1-0}{2^l-1} \leq 0.001$ => $l=10$ (pues $2^{10}-1=1023\geq1000$).
- **n:** $50$, igual que en el script de clase (`POP_SIZE`).
- **ps:** seleccion proporcional al fitness (ruleta), funcion `select` del script de clase.
- **pc:** $0.25$, el `CROSS_RATE` del script de clase.
- **pm:** $0.01$, el `MUTATION_RATE` del script de clase.
- **generaciones:** $150$, como en clase.

Abajo dejo el mismo esquema del programa de clase; solo cambio $F$, el dominio y $l$ segun la precision. El grafico de convergencia lo armo al final (en Colab la animacion `plt.ion` no se ve bien).

In [ ]:
# === programa GA de clase (adaptado a f(x)=3x^10) ===
# base: script_GA basico de Semana 5

DNA_SIZE = 10            # l segun precision 0.001 en [0,1]
POP_SIZE = 50            # n (como en clase)
CROSS_RATE = 0.25        # pc (como en clase)
MUTATION_RATE = 0.01     # pm (como en clase)
N_GENERATIONS = 150
X_BOUND = [0.0, 1.0]     # dominio asumido

def F(x):
    return 3.0 * (x ** 10)   # maximizar esto


# find non-zero fitness for selection
def get_fitness(pred):
    return pred + 1e-3 - np.min(pred)


# convert binary DNA to decimal and normalize to X_BOUND
# (misma idea del script de clase)
def translateDNA(pop):
    return pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2**DNA_SIZE - 1) * X_BOUND[1]


def select(pop, fitness):    # nature selection wrt pop's fitness
    idx = np.random.choice(np.arange(POP_SIZE), size=POP_SIZE, replace=True,
                           p=fitness / fitness.sum())
    return pop[idx]


def crossover(parent, pop):     # mating process (genes crossover)
    if np.random.rand() < CROSS_RATE:
        i_ = np.random.randint(0, POP_SIZE, size=1)
        cross_points = np.random.randint(0, 2, size=DNA_SIZE).astype(np.bool_)
        parent[cross_points] = pop[i_, cross_points]
    return parent


def mutate(child):
    for point in range(DNA_SIZE):
        if np.random.rand() < MUTATION_RATE:
            child[point] = 1 if child[point] == 0 else 0
    return child


pop = np.random.randint(2, size=(POP_SIZE, DNA_SIZE))

hist_mejor_f = []
hist_prom_f = []
hist_mejor_x = []

for _ in range(N_GENERATIONS):
    F_values = F(translateDNA(pop))

    fitness = get_fitness(F_values)
    i_best = np.argmax(fitness)

    hist_mejor_f.append(F_values[i_best])
    hist_prom_f.append(F_values.mean())
    hist_mejor_x.append(translateDNA(pop[i_best:i_best+1])[0])

    pop = select(pop, fitness)
    pop_copy = pop.copy()
    for parent in pop:
        child = crossover(parent, pop_copy)
        child = mutate(child)
        parent[:] = child

# resultado final
F_values = F(translateDNA(pop))
fitness = get_fitness(F_values)
i_best = np.argmax(fitness)
mejor_dna = pop[i_best, :]
mejor_x = translateDNA(pop[i_best:i_best+1])[0]
mejor_f = F_values[i_best]

print("Most fitted DNA:", mejor_dna)
print("Most fitted DNA real:", mejor_x)
print("f(x) =", mejor_f)
print("f(1) teorico =", F(1.0))
print("parametros: n=", POP_SIZE, "l=", DNA_SIZE, "pc=", CROSS_RATE, "pm=", MUTATION_RATE)

In [ ]:
# grafico de convergencia (lo que pide el enunciado)
gens = np.arange(1, N_GENERATIONS + 1)

plt.figure(figsize=(8, 4))
plt.plot(gens, hist_mejor_f, label="mejor f(x)")
plt.plot(gens, hist_prom_f, label="promedio f(x)")
plt.xlabel("generacion")
plt.ylabel("f(x)")
plt.title("Convergencia AG (script de clase) - f(x)=3x^10")
plt.legend()
plt.grid(True)
plt.show()

# como en clase: funcion + poblacion final
x = np.linspace(X_BOUND[0], X_BOUND[1], 200)
plt.figure(figsize=(8, 4))
plt.plot(x, F(x))
plt.scatter(translateDNA(pop), F(translateDNA(pop)), s=80, c="red", alpha=0.5)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Poblacion final sobre f(x)=3x^10")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(gens, hist_mejor_x)
plt.xlabel("generacion")
plt.ylabel("mejor x")
plt.title("Evolucion del mejor x")
plt.grid(True)
plt.show()

### Comentario pregunta 3

Con el mismo AG de clase (select / crossover / mutate), la mejor f sube hacia 3 y el mejor x se acerca a 1.
El promedio tambien sube, con un poco de ruido por la mutacion.

---
Luis Antonio Rojas Huaroc - 20171581B